# 37 — Autopilot quickstart

Autopilot is the **long-running, goal-driven, budget-gated** runner. It wraps `GoalAgent` + `PersistentAgent` with:

- **Goal-satisfaction termination** (not a fixed step count).
- **Budget gates**: wall-clock, tool calls, tokens, dollars, iterations.
- **Atomic checkpoints** so a crash costs at most one iteration.
- **Heartbeats + live event stream** for Claude-Desktop-style UIs.

This notebook runs against **Bedrock Llama 4 Scout** (the library default). Every cell below is safe to re-run in any order.


In [ ]:
from pathlib import Path
import sys

# Resolve the repo root whether this cell is running from ./notebooks
# or from the repo root — mirrors the existing 01–36 notebook series.
ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from examples.run_multi_tool_agent import build_llm_from_env

# Bedrock Llama 4 Scout is the default model this library ships with
# (see `shipit_agent/config.py`). No model name required — the helper
# reads `AWS_REGION` / credentials from env and returns an adapter that
# works with every Autopilot / Agent class.
llm = build_llm_from_env("bedrock")
print("Bedrock LLM ready:", type(llm).__name__)


## 1. Declare a Goal

A `Goal` is an objective plus a list of success criteria. Autopilot stops iterating the moment every criterion is satisfied OR a budget trips — whichever comes first.

In [ ]:
from shipit_agent.deep import Goal

goal = Goal(
    objective="Summarize the three most common Python dict gotchas and show a working snippet for each.",
    success_criteria=[
        "At least 3 gotchas explained in prose",
        "Each gotcha has a Python snippet that reproduces it",
        "A one-line fix or avoidance is shown for each",
    ],
)
goal


## 2. Build the Autopilot

`BudgetPolicy` has conservative defaults (30 min, 100 tool calls, \$5). For this demo we cap harder so it finishes quickly.

In [ ]:
from shipit_agent.autopilot import Autopilot, BudgetPolicy, default_heartbeat_stderr

autopilot = Autopilot(
    llm=llm,
    goal=goal,
    budget=BudgetPolicy(
        max_seconds=600,        # 10 min cap for the demo
        max_tool_calls=20,
        max_iterations=8,
        max_tokens=300_000,
        max_dollars=2.0,
    ),
    heartbeat_every_seconds=30,
    on_heartbeat=default_heartbeat_stderr,
)
print("Autopilot ready — budget:", autopilot.budget)


## 3. Run synchronously with `run(run_id=...)`

`run()` drains the goal loop and returns an `AutopilotResult`. Pass a stable `run_id` so the checkpoint is resumable later.

In [ ]:
result = autopilot.run(run_id="py-dict-gotchas-v1")

print(f"status:       {result.status}")
print(f"iterations:   {result.iterations}")
print(f"criteria met: {sum(1 for c in result.criteria_met if c)} / {len(result.criteria_met)}")
print(f"halt reason:  {result.halt_reason}")
print()
print(result.output[:800])


## 4. Inspect the checkpoint

Autopilot writes one JSON file per run to `~/.shipit_agent/checkpoints/<run_id>.json`. That file is enough to resume on another machine.

In [ ]:
import json, os
from pathlib import Path

cp = Path.home() / ".shipit_agent" / "checkpoints" / "py-dict-gotchas-v1.json"
if cp.exists():
    data = json.loads(cp.read_text())
    print(json.dumps({k: data[k] for k in ("run_id", "iterations", "usage")}, indent=2))
else:
    print("(no checkpoint — run cell 3 first)")


## 5. Understand `status`

- `completed` — every criterion passed.
- `partial` — some criteria passed; run halted because of budget.
- `halted` — budget tripped before any criterion could be verified.
- `failed` — an exception aborted the run; the checkpoint captures what was known at crash time.

You can call `autopilot.resume(run_id=...)` to pick up any non-`completed` run.

## Where next

- **38 — Autopilot live streaming** — watch the iteration, tool call, and heartbeat events as the run progresses.
- **39 — Persistence & the scheduler daemon** — fire-and-forget 24h operation.
- **40-42 — Specialist agents** — developer, debugger, researcher, design reviewer, PM, sales, CS, marketing.